## Обучение с учителем. Задача регрессии

In [23]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score
from math import sqrt

#### Чтение данных

In [2]:
data = pd.read_csv(f'../data/diamonds_filtered.csv')

data.head()

,carat,cut,color,clarity,depth,table,price,radius
0,0.29,4,2,4,62.4,58,334,2.100
1,0.31,2,1,2,63.3,58,335,2.170
2,0.30,2,1,3,64.0,55,339,2.125
3,0.31,5,1,2,62.2,54,344,2.175
4,0.32,4,6,1,60.9,58,345,2.190


#### Выделение целевого признака и предиктора

In [3]:
y = data['price']
x = data.drop(['price'], axis=1)

In [4]:
y[:5]

0    334
1    335
2    339
3    344
4    345
Name: price, dtype: int64

In [5]:
x[:5]

,carat,cut,color,clarity,depth,table,radius
0,0.29,4,2,4,62.4,58,2.100
1,0.31,2,1,2,63.3,58,2.170
2,0.30,2,1,3,64.0,55,2.125
3,0.31,5,1,2,62.2,54,2.175
4,0.32,4,6,1,60.9,58,2.190


#### Разделение данных на обучающую и тестовую выборки

In [6]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.15, random_state=81)

x_train

,carat,cut,color,clarity,depth,table,radius
16626,1.21,3,4,4,63.1,58,3.385
28087,0.30,5,5,4,61.6,57,2.165
28816,0.31,4,3,6,61.4,58,2.185
30464,0.41,5,5,4,60.8,56,2.400
38797,0.51,5,5,3,61.3,56,2.595
...,...,...,...,...,...,...,...
1009,0.81,4,5,3,61.3,60,3.000
3523,0.70,5,6,4,60.8,56,2.870
16331,1.03,2,6,5,63.3,59,3.200
4171,1.00,2,5,2,60.6,62,3.180


#### Масштабирование данных

In [7]:
scaler = StandardScaler()

x_train = scaler.fit_transform(x_train)     # считает среднее, отклонение и преобразует
x_test = scaler.transform(x_test)       # только преобразует (x - среднее) / отклонение

### Линейная регрессия

In [8]:
lr = LinearRegression()
lr.fit(x_train, y_train)

LinearRegression()

In [9]:
y_pred = lr.predict(x_test)

#### Метрики качества

In [10]:
print(f'MAE: {mean_absolute_error(y_test, y_pred)}')
print(f'MSE: {mean_squared_error(y_test, y_pred)}')
print(f'RMSE: {sqrt(mean_squared_error(y_test, y_pred))}')
print(f'MAPE: {sqrt(mean_absolute_percentage_error(y_test, y_pred))}')
print(f'R^2: {r2_score(y_test, y_pred)}')

MAE: 715.7342849673886
MSE: 1094671.312960932
RMSE: 1046.265412293139
MAPE: 0.6285430288129593
R^2: 0.9126883679986152


#### Значения весов

In [11]:
lr.coef_

array([ 5900.43180824,    57.86211668,   568.30857287,   779.09960723,
        -138.45111345,   -82.54400423, -2252.47496962])

### Lasso (L1)

#### Обучение Lasso с задаными параметрами

In [12]:
lasso = Lasso(alpha=0.5, max_iter=10_000).fit(x_train, y_train)
y_pred = lasso.predict(x_test)

#### Метрики качества

In [13]:
print(f'MAE: {mean_absolute_error(y_test, y_pred)}')
print(f'MSE: {mean_squared_error(y_test, y_pred)}')
print(f'RMSE: {sqrt(mean_squared_error(y_test, y_pred))}')
print(f'MAPE: {sqrt(mean_absolute_percentage_error(y_test, y_pred))}')
print(f'R^2: {lasso.score(x_test, y_test)}')

MAE: 716.0775706165575
MSE: 1094883.6669045647
RMSE: 1046.3668892432352
MAPE: 0.6287908824030116
R^2: 0.9126714305223509


#### Подбор гиперпараметра при помощи `GridSearchCV`

In [14]:
params = {
    'alpha' : np.arange(0.1, 1.1, 0.1)
}

params

{'alpha': array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1. ])}

In [ ]:
grid_lasso = GridSearchCV(
    lasso,
    params,
    cv = 5,
)

grid_lasso.fit(x_train, y_train)
best_alpha = grid_lasso.best_params_['alpha']

print(f'Оптимальное значение гиперпараметра alpha = {best_alpha}')

Оптимальное значение гиперпараметра альфа = 0.1


In [19]:
y_pred = grid_lasso.predict(x_test)

#### Метрики качества

In [20]:
print(f'MAE: {mean_absolute_error(y_test, y_pred)}')
print(f'MSE: {mean_squared_error(y_test, y_pred)}')
print(f'RMSE: {sqrt(mean_squared_error(y_test, y_pred))}')
print(f'MAPE: {sqrt(mean_absolute_percentage_error(y_test, y_pred))}')
print(f'R^2: {grid_lasso.score(x_test, y_test)}')

MAE: 715.7998423202026
MSE: 1094707.7331420965
RMSE: 1046.2828169964832
MAPE: 0.6285910165387173
R^2: 0.9126854631034036


#### Подбор гиперпараметра при помощи `RandomizedSearchCV`

In [22]:
params = {'alpha' : np.arange(0.01, 1.1, 0.01)}
params

{'alpha': array([0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1 , 0.11,
        0.12, 0.13, 0.14, 0.15, 0.16, 0.17, 0.18, 0.19, 0.2 , 0.21, 0.22,
        0.23, 0.24, 0.25, 0.26, 0.27, 0.28, 0.29, 0.3 , 0.31, 0.32, 0.33,
        0.34, 0.35, 0.36, 0.37, 0.38, 0.39, 0.4 , 0.41, 0.42, 0.43, 0.44,
        0.45, 0.46, 0.47, 0.48, 0.49, 0.5 , 0.51, 0.52, 0.53, 0.54, 0.55,
        0.56, 0.57, 0.58, 0.59, 0.6 , 0.61, 0.62, 0.63, 0.64, 0.65, 0.66,
        0.67, 0.68, 0.69, 0.7 , 0.71, 0.72, 0.73, 0.74, 0.75, 0.76, 0.77,
        0.78, 0.79, 0.8 , 0.81, 0.82, 0.83, 0.84, 0.85, 0.86, 0.87, 0.88,
        0.89, 0.9 , 0.91, 0.92, 0.93, 0.94, 0.95, 0.96, 0.97, 0.98, 0.99,
        1.  , 1.01, 1.02, 1.03, 1.04, 1.05, 1.06, 1.07, 1.08, 1.09])}

In [24]:
random_lasso = RandomizedSearchCV(
    lasso,
    params,
    cv = 5
)

random_lasso.fit(x_train, y_train)

best_alpha = random_lasso.best_params_['alpha']

print(f'Оптимальное значение гиперпараметра alpha = {best_alpha}')

Оптимальное значение гиперпараметра alpha = 0.02


In [25]:
y_pred = random_lasso.predict(x_test)

#### Метрики качества

In [26]:
print(f'MAE: {mean_absolute_error(y_test, y_pred)}')
print(f'MSE: {mean_squared_error(y_test, y_pred)}')
print(f'RMSE: {sqrt(mean_squared_error(y_test, y_pred))}')
print(f'MAPE: {sqrt(mean_absolute_percentage_error(y_test, y_pred))}')
print(f'R^2: {random_lasso.score(x_test, y_test)}')

MAE: 715.7473111856287
MSE: 1094678.350998761
RMSE: 1046.268775697125
MAPE: 0.6285525956033053
R^2: 0.9126878066405513


### Ridge (L2)